In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

# Giả định compute_eer, compute_mindcf ở trong file metrics
from metrics import compute_eer, compute_mindcf

# TODO: Bạn cần import class Model thực tế của bạn vào đây
# from your_model_file import YourSpeakerVerificationModel 

# ==============================================================================
# CẤU HÌNH THÔNG SỐ VÀ ĐƯỜNG DẪN 
# ==============================================================================
# 1. Chế độ chạy (Cấu hình Mode)
# MODE = 1: PTM Only
# MODE = 2: Handcrafted Only
# MODE = 3: Fusion Mode
MODE = 3 
FUSION_METHOD = 'cross_attention' # Dùng cho Mode 3: 'concat', 'cross_attention', 'gating'

# 2. Đường dẫn (BẠN ĐIỀN ĐƯỜNG DẪN CỦA MÌNH VÀO ĐÂY)
CSV_FILE_PATH = r"D:\Study\7-SP26\DATxSLP\test_list_gt.csv" 

# Trỏ đến TỆP .pt cụ thể chứa Dictionary đặc trưng của model/phương pháp bạn muốn test
PTM_DICT_PATH = r"ĐƯỜNG_DẪN_TỚI_FILE_PTM.pt"                   # VD: folder_PTM/wavlm.pt
HANDCRAFTED_DICT_PATH = r"ĐƯỜNG_DẪN_TỚI_FILE_HANDCRAFTED.pt"   # VD: folder_HC/MFCC_Pitch.pt

# Trỏ đến file trọng số của mô hình đã train tương ứng với Mode đang chọn
MODEL_WEIGHTS_PATH = r"ĐƯỜNG_DẪN_TỚI_TRỌNG_SỐ_MODEL.pt"

P_TARGET = 0.05
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# KHỞI TẠO MÔ HÌNH VÀ TẢI DỮ LIỆU
# ==============================================================================
print(f"--- ĐANG CHẠY MODE {MODE} ---")

# 1. Khởi tạo mô hình (Giả lập việc khởi tạo, bạn thay bằng code thật của bạn)
# model = YourSpeakerVerificationModel(mode=MODE, fusion_method=FUSION_METHOD).to(DEVICE)
# model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
# model.eval() 

# 2. Tải các file đặc trưng (.pt dictionaries) dựa trên Mode
ptm_dict = {}
hc_dict = {}

if MODE in [1, 3]:
    print(f"Loading PTM features từ: {os.path.basename(PTM_DICT_PATH)}...")
    ptm_dict = torch.load(PTM_DICT_PATH)
    
if MODE in [2, 3]:
    print(f"Loading Handcrafted features từ: {os.path.basename(HANDCRAFTED_DICT_PATH)}...")
    hc_dict = torch.load(HANDCRAFTED_DICT_PATH)

# ==============================================================================
# QUY TRÌNH TRÍCH XUẤT VÀ ĐÁNH GIÁ CẶP
# ==============================================================================
print(f"\n1. Đang phân tích file CSV: {os.path.basename(CSV_FILE_PATH)}...")
with open(CSV_FILE_PATH, 'r') as f:
    lines = f.readlines()

# Lấy danh sách các file duy nhất để tiết kiệm thời gian forward qua mô hình
unique_files = set()
pairs = []

for line in lines:
    parts = line.strip().split()
    if len(parts) != 3:
        continue
    label = int(parts[0])
    file1 = os.path.normpath(parts[1])
    file2 = os.path.normpath(parts[2])
    unique_files.update([file1, file2])
    pairs.append((label, file1, file2))

print(f"-> Tìm thấy {len(pairs)} cặp và {len(unique_files)} file audio duy nhất.")

# Dictionary mới để chứa Speaker Embedding cuối cùng sau khi đi qua ECAPATDNN
final_embeddings_dict = {}
missing_files = 0

print("\n2. Đang tạo Speaker Embeddings qua mô hình...")
# with torch.no_grad(): # Bỏ comment khi chạy mô hình thật
for file_path in tqdm(unique_files, desc="Extracting embeddings"):
    try:
        # Lấy input feature tùy theo Mode
        if MODE == 1:
            feat = ptm_dict[file_path].to(DEVICE)
            # embedding = model(feat)  # Inference qua model
            embedding = feat # DÒNG GIẢ LẬP: Chờ bạn gắn model thật
            
        elif MODE == 2:
            feat = hc_dict[file_path].to(DEVICE)
            # embedding = model(feat)  # Inference qua model
            embedding = feat # DÒNG GIẢ LẬP: Chờ bạn gắn model thật
            
        elif MODE == 3:
            feat_ptm = ptm_dict[file_path].to(DEVICE)
            feat_hc = hc_dict[file_path].to(DEVICE)
            # embedding = model(feat_ptm, feat_hc)  # Inference qua model
            embedding = feat_ptm # DÒNG GIẢ LẬP: Chờ bạn gắn model thật
            
        final_embeddings_dict[file_path] = embedding.cpu()
    except KeyError:
        missing_files += 1

if missing_files > 0:
    print(f"Cảnh báo: Không tìm thấy đặc trưng cho {missing_files} file.")

# ==============================================================================
# TÍNH COSINE SIMILARITY
# ==============================================================================
print("\n3. Bắt đầu tính điểm Similarity cho các cặp...")
y_true = []
scores = []
missing_pairs = 0

for label, file1, file2 in tqdm(pairs, desc="Đang đánh giá"):
    if file1 not in final_embeddings_dict or file2 not in final_embeddings_dict:
        missing_pairs += 1
        continue
        
    emb1 = final_embeddings_dict[file1].view(-1)
    emb2 = final_embeddings_dict[file2].view(-1)
    
    score = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()
    
    y_true.append(label)
    scores.append(score)

y_true = np.array(y_true)
scores = np.array(scores)

print(f"\n-> Đã xử lý xong {len(y_true)} cặp. Bỏ qua {missing_pairs} cặp do thiếu dữ liệu.")

# ==============================================================================
# KẾT QUẢ ĐÁNH GIÁ METRIC 
# ==============================================================================
if len(y_true) > 0:
    print("\n4. Tính toán EER và MinDCF...")
    eer, eer_thresh = compute_eer(y_true, scores)
    min_dcf, dcf_thresh = compute_mindcf(y_true, scores, p_target=P_TARGET)
    
    print("\n" + "="*50)
    print(f" KẾT QUẢ ĐÁNH GIÁ (TEST SET) - MODE {MODE}")
    if MODE == 3:
        print(f" FUSION METHOD: {FUSION_METHOD.upper()}")
    print("="*50)
    print(f" Tổng số cặp (Pairs) : {len(y_true):,}")
    print(f" EER (%)             : {eer * 100:.2f} %")
    print(f" EER Threshold       : {eer_thresh:.4f}")
    print(f" MinDCF (p={P_TARGET})     : {min_dcf:.4f}")
    print(f" MinDCF Threshold    : {dcf_thresh:.4f}")
    print("="*50)
else:
    print("Không có cặp nào được tính cả. Hãy kiểm tra lại các file dictionary đặc trưng.")